# 02. In-Language Segmentation

**Paper section:** §4 In-language segmentation (Table 2).
**What it computes:** For each of Akkadian, Sumerian, and Elamite, learns a transitional-probability segmenter on segmented documents and reports per-language F1 / precision / recall at the optimal threshold. Compares against a Morfessor baseline. Includes a robustness sweep over the Morfessor `corpusweight` hyperparameter.
**Inputs:** Outputs of notebook 01.
**Outputs:** `outputs/table2_in_language_f1.csv`, `outputs/threshold_sweep_{lang}.csv` (one per language), `outputs/morfessor_corpusweight_sweep.json`.
**Expected runtime (CPU baseline):** ~5 min on a laptop CPU.

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)
doc_corpora = corpora['_documents']


## Experiment 2: Word Boundary Inference

Transitional probability on unsegmented Unicode sign streams.  
Gold boundaries from known word-segmented documents.

In [ ]:
# ============================================================
# EXPERIMENT 2: WORD BOUNDARY INFERENCE
# ============================================================
exp2_results = {}

for lang, df in datasets.items():
    print(f"\n--- Word Boundaries: {lang.upper()} ---")
    docs_unicode = doc_corpora[lang]['unicode']

    gold_data = []
    for doc in docs_unicode.values():
        if doc and len(doc.split()) > 2:
            boundaries, pos = [], 0
            for ch in doc:
                if ch == ' ': boundaries.append(pos)
                else: pos += 1
            continuous = doc.replace(' ', '')
            if len(continuous) > 3 and boundaries:
                gold_data.append({'segmented': doc, 'continuous': continuous, 'boundaries': boundaries})

    if len(gold_data) < 5:
        print(f"  Insufficient docs ({len(gold_data)})")
        exp2_results[lang] = {'lang': lang, 'note': 'insufficient_data'}
        continue

    print(f"  Gold documents: {len(gold_data)}")
    all_cont = ''.join(d['continuous'] for d in gold_data)
    unigrams = Counter(all_cont)
    bigrams = Counter(all_cont[i:i+2] for i in range(len(all_cont)-1))

    def tp(c1, c2):
        return bigrams[c1+c2] / unigrams[c1] if unigrams[c1] > 0 else 0

    best_f1, best_thresh = 0, 0
    for thresh in np.arange(0.05, 0.95, 0.05):
        tp_t, fp_t, fn_t = 0, 0, 0
        for d in gold_data:
            text = d['continuous']; gold_b = set(d['boundaries'])
            pred_b = {i for i in range(1, len(text)-1) if tp(text[i-1], text[i]) < thresh}
            tp_t += len(gold_b & pred_b); fp_t += len(pred_b - gold_b); fn_t += len(gold_b - pred_b)
        p = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
        r = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0
        f1 = 2*p*r/(p+r) if (p+r) else 0
        if f1 > best_f1: best_f1, best_thresh = f1, thresh

    # Final metrics at best threshold
    tp_t, fp_t, fn_t = 0, 0, 0
    for d in gold_data:
        text = d['continuous']; gold_b = set(d['boundaries'])
        pred_b = {i for i in range(1, len(text)-1) if tp(text[i-1], text[i]) < best_thresh}
        tp_t += len(gold_b & pred_b); fp_t += len(pred_b - gold_b); fn_t += len(gold_b - pred_b)
    precision = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
    recall = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0

    exp2_results[lang] = {
        'lang': lang, 'tp_f1': best_f1, 'tp_precision': precision,
        'tp_recall': recall, 'tp_threshold': best_thresh, 'n_docs': len(gold_data)
    }
    print(f"  TP: F1={best_f1:.4f} P={precision:.4f} R={recall:.4f} thresh={best_thresh:.2f}")

In [ ]:
# Install cunei-tools from GitHub
!pip install git+https://anonymous.4open.science/r/cunei-tools

from cunei_tools import CuneiSeg
from sklearn.model_selection import KFold
import numpy as np

for lang in ['akk', 'sux', 'elx']:
    docs = list(doc_corpora[lang]['unicode'].values())
    docs = [d for d in docs if d and len(d.split()) > 2]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1s = []

    for train_idx, test_idx in kf.split(docs):
        train_docs = [docs[i] for i in train_idx]
        test_docs = [docs[i] for i in test_idx]

        seg = CuneiSeg(lang=lang)
        seg.train(train_docs)
        metrics = seg.find_optimal_threshold(test_docs)
        fold_f1s.append(metrics['f1'])

    print(f"{lang.upper()}: held-out F1 = {np.mean(fold_f1s):.4f} (±{np.std(fold_f1s):.4f})")

In [ ]:
# Export threshold sweep data for figures
import csv

for lang in ['akk', 'sux', 'elx']:
    docs_unicode = doc_corpora[lang]['unicode']
    gold_data = []
    for doc in docs_unicode.values():
        if doc and len(doc.split()) > 2:
            boundaries, pos = [], 0
            for ch in doc:
                if ch == ' ': boundaries.append(pos)
                else: pos += 1
            continuous = doc.replace(' ', '')
            if len(continuous) > 3 and boundaries:
                gold_data.append({'continuous': continuous, 'boundaries': boundaries})

    all_cont = ''.join(d['continuous'] for d in gold_data)
    unigrams = Counter(all_cont)
    bigrams = Counter(all_cont[i:i+2] for i in range(len(all_cont)-1))

    rows = []
    for thresh in np.arange(0.05, 0.96, 0.05):
        tp_t, fp_t, fn_t = 0, 0, 0
        for d in gold_data:
            text = d['continuous']; gold_b = set(d['boundaries'])
            pred_b = {i for i in range(1, len(text)-1)
                      if bigrams[text[i-1]+text[i]] / unigrams[text[i-1]] < thresh
                      if unigrams[text[i-1]] > 0}
            tp_t += len(gold_b & pred_b)
            fp_t += len(pred_b - gold_b)
            fn_t += len(gold_b - pred_b)
        p = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
        r = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0
        f1 = 2*p*r/(p+r) if (p+r) else 0
        rows.append({'threshold': thresh, 'f1': f1, 'precision': p, 'recall': r})

    pd.DataFrame(rows).to_csv(f'threshold_sweep_{lang}.csv', index=False)
    print(f"{lang}: saved {len(rows)} threshold points")

In [ ]:
!pip install morfessor --quiet

In [ ]:
# ============================================================
# MORFESSOR BASELINE: 5-FOLD HELD-OUT CV
# Mirrors the structure of the existing CuneiSeg experiment so that
# numbers are directly comparable to Table 2 of the paper.
# ============================================================
from collections import Counter
import numpy as np
from sklearn.model_selection import KFold
import morfessor


def _train_morfessor(train_docs, corpusweight=1.0):
    word_counts = Counter()
    for doc in train_docs:
        for w in doc.split():
            if w:
                word_counts[w] += 1
    model = morfessor.BaselineModel(corpusweight=corpusweight)
    model.load_data([(c, w) for w, c in word_counts.items()])
    model.train_batch()
    return model, word_counts


def _predict_boundaries(model, continuous):
    if not continuous:
        return set()
    try:
        segs, _ = model.viterbi_segment(continuous)
    except Exception:
        return set()
    b, cur = set(), 0
    for s in segs[:-1]:
        cur += len(s)
        b.add(cur)
    return b


def _gold_boundaries(segmented):
    g, pos = set(), 0
    for ch in segmented:
        if ch == ' ':
            g.add(pos)
        else:
            pos += 1
    return g


def run_morfessor_cv(doc_corpora, langs=('akk', 'sux', 'elx'),
                    corpusweight=1.0, n_splits=5, random_state=42):
    results = {}
    for lang in langs:
        if lang not in doc_corpora:
            continue
        docs = [d for d in doc_corpora[lang]['unicode'].values()
                if d and len(d.split()) > 2]
        if len(docs) < n_splits:
            print(f"{lang.upper()}: only {len(docs)} docs, skipping")
            continue

        print(f"\n--- Morfessor: {lang.upper()} "
              f"({len(docs)} docs, corpusweight={corpusweight}) ---")
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        fold_metrics = []
        for fi, (tr, te) in enumerate(kf.split(docs)):
            train = [docs[i] for i in tr]
            test = [docs[i] for i in te]
            model, _ = _train_morfessor(train, corpusweight=corpusweight)
            tp_n, fp_n, fn_n = 0, 0, 0
            for d in test:
                if not d or len(d.split()) < 2:
                    continue
                gold = _gold_boundaries(d)
                pred = _predict_boundaries(model, d.replace(' ', ''))
                tp_n += len(pred & gold)
                fp_n += len(pred - gold)
                fn_n += len(gold - pred)
            p = tp_n / (tp_n + fp_n) if (tp_n + fp_n) else 0.0
            r = tp_n / (tp_n + fn_n) if (tp_n + fn_n) else 0.0
            f1 = 2 * p * r / (p + r) if (p + r) else 0.0
            fold_metrics.append({'p': p, 'r': r, 'f1': f1})
            print(f"  fold {fi+1}: F1={f1:.4f} P={p:.4f} R={r:.4f}")

        f1s = [m['f1'] for m in fold_metrics]
        ps = [m['p'] for m in fold_metrics]
        rs = [m['r'] for m in fold_metrics]
        results[lang] = {
            'f1_mean': float(np.mean(f1s)),
            'f1_std': float(np.std(f1s)),
            'precision_mean': float(np.mean(ps)),
            'recall_mean': float(np.mean(rs)),
            'corpusweight': corpusweight,
            'folds': fold_metrics,
        }
        print(f"  {lang.upper()}: F1 = {np.mean(f1s):.4f} (±{np.std(f1s):.4f}), "
              f"P = {np.mean(ps):.4f}, R = {np.mean(rs):.4f}")
    return results


# Run with default corpusweight = 1.0 (Morfessor's published default)
morfessor_results = run_morfessor_cv(doc_corpora)

### Robustness sweep over corpusweight

Pre-empts the reviewer concern that the baseline was under-tuned. Reports
Morfessor's best F1 across `corpusweight ∈ {0.25, 0.5, 1.0, 2.0, 4.0}`.
Lower corpusweight discourages segmentation (fewer boundaries, higher
precision); higher encourages it (more boundaries, higher recall).

In [ ]:
sweep_results = {}
for cw in [0.25, 0.5, 1.0, 2.0, 4.0]:
    print(f"\n########## corpusweight = {cw} ##########")
    sweep_results[cw] = run_morfessor_cv(doc_corpora, corpusweight=cw)

# Best per language across sweep
print("\n\n========== BEST PER LANGUAGE ACROSS SWEEP ==========")
for lang in ['akk', 'sux', 'elx']:
    best_cw, best_f1 = None, -1
    for cw, res in sweep_results.items():
        if lang in res and res[lang]['f1_mean'] > best_f1:
            best_f1 = res[lang]['f1_mean']
            best_cw = cw
    if best_cw is not None:
        m = sweep_results[best_cw][lang]
        print(f"  {lang.upper()}: best F1 = {m['f1_mean']:.4f} "
              f"(±{m['f1_std']:.4f}) at corpusweight={best_cw}, "
              f"P={m['precision_mean']:.4f}, R={m['recall_mean']:.4f}")

In [ ]:
# Compose a paper-ready table line for the LaTeX update to Table 2
print("\nLaTeX table row (for Table 2 update):\n")
print("Prior work \\citep{homburg-chiarcos-2016}: \\\\")
print("MaxMatch & .739 & --- & --- & --- \\\\")
print("Bigram & .149 & --- & --- & --- \\\\")
print("\nNew baseline (Morfessor, this work): \\\\")
for lang in ['akk', 'sux', 'elx']:
    if lang in morfessor_results:
        m = morfessor_results[lang]
        name = {'akk':'Akkadian','sux':'Sumerian','elx':'Elamite'}[lang]
        print(f"Morfessor ({name}) & {m['f1_mean']:.3f} & "
              f"{m['precision_mean']:.3f} & {m['recall_mean']:.3f} & --- \\\\")

In [ ]:
# ============================================================
# MORFESSOR CROSS-LANGUAGE TRANSFER
# Train on one language, test on another
# Compare directly to TP transfer results in Table 3
# ============================================================
import morfessor
from collections import Counter
import numpy as np


def train_morfessor_full(docs, corpusweight=1.0):
    """Train Morfessor on ALL documents of a language."""
    word_counts = Counter()
    for d in docs:
        for w in d.split():
            if w:
                word_counts[w] += 1
    model = morfessor.BaselineModel(corpusweight=corpusweight)
    model.load_data([(c, w) for w, c in word_counts.items()])
    model.train_batch()
    return model


def predict_boundaries(model, continuous):
    if not continuous:
        return set()
    try:
        segs, _ = model.viterbi_segment(continuous)
    except Exception:
        return set()
    b, cur = set(), 0
    for s in segs[:-1]:
        cur += len(s)
        b.add(cur)
    return b


def gold_boundaries(segmented):
    g, pos = set(), 0
    for ch in segmented:
        if ch == ' ':
            g.add(pos)
        else:
            pos += 1
    return g


def evaluate_on_lang(model, test_docs):
    """Evaluate a trained model on test documents."""
    tp_n, fp_n, fn_n = 0, 0, 0
    for d in test_docs:
        if not d or len(d.split()) < 2:
            continue
        gold = gold_boundaries(d)
        pred = predict_boundaries(model, d.replace(' ', ''))
        tp_n += len(pred & gold)
        fp_n += len(pred - gold)
        fn_n += len(gold - pred)
    p = tp_n / (tp_n + fp_n) if (tp_n + fp_n) else 0.0
    r = tp_n / (tp_n + fn_n) if (tp_n + fn_n) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {'f1': f1, 'precision': p, 'recall': r}


# ── Build document lists ──
langs = ['akk', 'sux', 'elx']
lang_names = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite'}
lang_docs = {}
for lang in langs:
    if lang in doc_corpora:
        lang_docs[lang] = [d for d in doc_corpora[lang]['unicode'].values()
                           if d and len(d.split()) > 2]
        print(f"{lang.upper()}: {len(lang_docs[lang])} docs")

# ── Train one model per language ──
models = {}
for lang in langs:
    if lang in lang_docs:
        print(f"\nTraining Morfessor on {lang.upper()}...")
        models[lang] = train_morfessor_full(lang_docs[lang])

# ── Full 3×3 transfer matrix ──
print(f"\n{'='*60}")
print(f"  MORFESSOR CROSS-LANGUAGE TRANSFER")
print(f"{'='*60}")
print(f"\n  {'Train':>12s} {'Test':>12s} {'F1':>8s} {'P':>8s} {'R':>8s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8}")

transfer = {}
for train_lang in langs:
    for test_lang in langs:
        if train_lang not in models or test_lang not in lang_docs:
            continue
        metrics = evaluate_on_lang(models[train_lang], lang_docs[test_lang])
        key = f"{train_lang}→{test_lang}"
        transfer[key] = metrics
        marker = "  ← same" if train_lang == test_lang else ""
        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{metrics['f1']:>8.4f} {metrics['precision']:>8.4f} "
              f"{metrics['recall']:>8.4f}{marker}")

# ── Compare to TP ──
tp_transfer = {
    'akk→akk': 0.971, 'akk→sux': 0.968, 'akk→elx': 0.995,
    'sux→akk': 0.966, 'sux→sux': 0.972, 'sux→elx': 0.995,
    'elx→akk': 0.959, 'elx→sux': 0.969, 'elx→elx': 0.989,
}

print(f"\n{'='*60}")
print(f"  TP vs MORFESSOR COMPARISON")
print(f"{'='*60}")
print(f"\n  {'Pair':>12s} {'TP F1':>8s} {'Morf F1':>8s} {'Winner':>10s}")
print(f"  {'-'*12} {'-'*8} {'-'*8} {'-'*10}")

for key in sorted(tp_transfer.keys()):
    tp_f1 = tp_transfer[key]
    morf_f1 = transfer.get(key, {}).get('f1', 0)
    winner = "TP" if tp_f1 > morf_f1 else "Morfessor" if morf_f1 > tp_f1 else "Tie"
    same = " (same)" if key.split('→')[0] == key.split('→')[1] else ""
    print(f"  {key:>12s} {tp_f1:>8.3f} {morf_f1:>8.4f} {winner:>10s}{same}")

In [ ]:
# ============================================================
# MORFESSOR CROSS-LANGUAGE TRANSFER (revised)
# Train Morfessor on all docs of one language, evaluate on all
# docs of another. Off-diagonal cells only: diagonal numbers
# come from the held-out 5-fold CV run above, so we do not
# evaluate in-sample.
#
# Output structure matches Table 3 of the paper:
#   - in-language reference values (CV held-out)
#   - cross-language transfer (zero-shot, no test-language tuning)
# Saves results to JSON for backup.
# ============================================================
import json
import morfessor
import numpy as np
from collections import Counter


# ---------- helpers (same as CV experiment) ----------

def _train_morfessor_full(docs, corpusweight=1.0):
    """Train Morfessor on word frequencies from ALL given docs."""
    word_counts = Counter()
    for d in docs:
        for w in d.split():
            if w:
                word_counts[w] += 1
    model = morfessor.BaselineModel(corpusweight=corpusweight)
    model.load_data([(c, w) for w, c in word_counts.items()])
    model.train_batch()
    return model, word_counts


def _predict_boundaries(model, continuous):
    if not continuous:
        return set()
    try:
        segs, _ = model.viterbi_segment(continuous)
    except Exception:
        return set()
    b, cur = set(), 0
    for s in segs[:-1]:
        cur += len(s)
        b.add(cur)
    return b


def _gold_boundaries(segmented):
    g, pos = set(), 0
    for ch in segmented:
        if ch == ' ':
            g.add(pos)
        else:
            pos += 1
    return g


def _evaluate(model, test_docs):
    tp_n, fp_n, fn_n = 0, 0, 0
    for d in test_docs:
        if not d or len(d.split()) < 2:
            continue
        gold = _gold_boundaries(d)
        pred = _predict_boundaries(model, d.replace(' ', ''))
        tp_n += len(pred & gold)
        fp_n += len(pred - gold)
        fn_n += len(gold - pred)
    p = tp_n / (tp_n + fp_n) if (tp_n + fp_n) else 0.0
    r = tp_n / (tp_n + fn_n) if (tp_n + fn_n) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {'f1': f1, 'precision': p, 'recall': r,
            'tp': tp_n, 'fp': fp_n, 'fn': fn_n}


def _vocab_overlap(train_counts, test_docs):
    """Fraction of test tokens whose word form appears in train vocab.
    Useful to characterize WHY transfer fails: lexicon mismatch."""
    train_vocab = set(train_counts.keys())
    seen, total = 0, 0
    for d in test_docs:
        for w in d.split():
            if w:
                total += 1
                if w in train_vocab:
                    seen += 1
    return seen / total if total else 0.0


# ---------- build doc lists ----------

LANGS = ['akk', 'sux', 'elx']
LANG_NAMES = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite'}

lang_docs = {}
for lang in LANGS:
    if lang in doc_corpora:
        lang_docs[lang] = [d for d in doc_corpora[lang]['unicode'].values()
                           if d and len(d.split()) > 2]
        print(f"{lang.upper()}: {len(lang_docs[lang])} docs")


# ---------- train one model per source language ----------

print("\nTraining one Morfessor model per language on ALL docs...")
models, vocabs = {}, {}
for lang in LANGS:
    if lang in lang_docs:
        print(f"  training {lang.upper()}...")
        models[lang], vocabs[lang] = _train_morfessor_full(lang_docs[lang])
        print(f"    vocab size: {len(vocabs[lang])} types")


# ---------- diagonal: use CV numbers from earlier held-out experiment ----------
# Pulled from morfessor_results if available; falls back to your CV output.
try:
    morf_diagonal_cv = {
        lang: {
            'f1': morfessor_results[lang]['f1_mean'],
            'precision': morfessor_results[lang]['precision_mean'],
            'recall': morfessor_results[lang]['recall_mean'],
        }
        for lang in LANGS if lang in morfessor_results
    }
    print("\nDiagonal CV numbers loaded from earlier morfessor_results.")
except NameError:
    morf_diagonal_cv = {
        'akk': {'f1': 0.9964, 'precision': 0.9971, 'recall': 0.9957},
        'sux': {'f1': 0.9930, 'precision': 0.9967, 'recall': 0.9895},
        'elx': {'f1': 0.9846, 'precision': 0.9994, 'recall': 0.9705},
    }
    print("\nDiagonal CV numbers loaded from hardcoded fallback.")


# ---------- off-diagonal: cross-language transfer ----------

print("\n" + "=" * 72)
print("  MORFESSOR CROSS-LANGUAGE TRANSFER (zero-shot, off-diagonal)")
print("=" * 72)
header = f"  {'Train':>10s} {'Test':>10s} {'F1':>8s} {'P':>8s} {'R':>8s} {'VocOverlap':>12s}"
print(header)
print("  " + "-" * (len(header) - 2))

transfer = {}
for train_lang in LANGS:
    if train_lang not in models:
        continue
    for test_lang in LANGS:
        if test_lang not in lang_docs:
            continue
        if train_lang == test_lang:
            # Skip diagonal: report CV number from the held-out experiment
            d = morf_diagonal_cv[train_lang]
            transfer[f"{train_lang}->{train_lang}"] = {
                **d, 'note': 'CV held-out (from previous run)', 'vocab_overlap': 1.0,
            }
            print(f"  {LANG_NAMES[train_lang]:>10s} {LANG_NAMES[test_lang]:>10s} "
                  f"{d['f1']:>8.4f} {d['precision']:>8.4f} {d['recall']:>8.4f} "
                  f"{'(CV)':>12s}")
            continue
        m = _evaluate(models[train_lang], lang_docs[test_lang])
        overlap = _vocab_overlap(vocabs[train_lang], lang_docs[test_lang])
        m['vocab_overlap'] = overlap
        transfer[f"{train_lang}->{test_lang}"] = m
        print(f"  {LANG_NAMES[train_lang]:>10s} {LANG_NAMES[test_lang]:>10s} "
              f"{m['f1']:>8.4f} {m['precision']:>8.4f} {m['recall']:>8.4f} "
              f"{overlap:>12.4f}")


# ---------- side-by-side with TP ----------
# TP top panel: threshold tuned on test (per Table 3 top of paper).
# TP bottom panel: zero-shot, no tuning (per Table 3 bottom of paper).

tp_tuned = {
    'akk->akk': 0.971, 'akk->sux': 0.968, 'akk->elx': 0.995,
    'sux->akk': 0.966, 'sux->sux': 0.972, 'sux->elx': 0.995,
    'elx->akk': 0.959, 'elx->sux': 0.969, 'elx->elx': 0.989,
}
tp_zeroshot = {
    'akk->sux': 0.967, 'akk->elx': 0.992,
    'sux->akk': 0.960, 'sux->elx': 0.993,
}

print("\n" + "=" * 72)
print("  TP vs MORFESSOR (zero-shot comparison)")
print("=" * 72)
print(f"\n  {'Pair':>10s} {'TP zero-shot':>14s} {'Morfessor':>11s} "
      f"{'Δ (TP-Morf)':>14s} {'Winner':>10s}")
print("  " + "-" * 64)
for key in ['akk->sux', 'akk->elx', 'sux->akk', 'sux->elx']:
    tp_v = tp_zeroshot[key]
    mf_v = transfer.get(key, {}).get('f1', 0.0)
    delta = tp_v - mf_v
    winner = 'TP' if tp_v > mf_v else 'Morf' if mf_v > tp_v else 'Tie'
    print(f"  {key:>10s} {tp_v:>14.4f} {mf_v:>11.4f} {delta:>+14.4f} {winner:>10s}")

# Also show ELX-as-source cells for completeness (no TP zero-shot baseline)
print(f"\n  ELX-as-source cells (TP zero-shot not reported in paper):")
for key in ['elx->akk', 'elx->sux']:
    mf_v = transfer.get(key, {}).get('f1', 0.0)
    print(f"    {key}: Morfessor F1 = {mf_v:.4f}")


# ---------- summary stats useful for paper writing ----------

off_diag = [v['f1'] for k, v in transfer.items() if k.split('->')[0] != k.split('->')[1]]
diag = [v['f1'] for k, v in transfer.items() if k.split('->')[0] == k.split('->')[1]]

print("\n" + "=" * 72)
print("  HEADLINE NUMBERS FOR THE PAPER")
print("=" * 72)
print(f"  Morfessor in-language (CV, mean across langs): {np.mean(diag):.4f}")
print(f"  Morfessor cross-language (mean across pairs):  {np.mean(off_diag):.4f}")
print(f"  Morfessor cross-language (range):              "
      f"{min(off_diag):.4f} to {max(off_diag):.4f}")
print(f"  Cross-language degradation (mean):             "
      f"{np.mean(diag) - np.mean(off_diag):+.4f}")

# TP comparison
tp_off_diag = list(tp_zeroshot.values())
print(f"\n  TP cross-language (zero-shot, mean across pairs): {np.mean(tp_off_diag):.4f}")
print(f"  TP cross-language (zero-shot, range):             "
      f"{min(tp_off_diag):.4f} to {max(tp_off_diag):.4f}")


# ---------- save to JSON ----------

out = {
    'morfessor_transfer': transfer,
    'morfessor_diagonal_cv': morf_diagonal_cv,
    'tp_tuned_reference': tp_tuned,
    'tp_zeroshot_reference': tp_zeroshot,
    'vocab_sizes': {lang: len(vocabs[lang]) for lang in vocabs},
    'doc_counts': {lang: len(docs) for lang, docs in lang_docs.items()},
    'summary': {
        'morf_diag_mean': float(np.mean(diag)),
        'morf_offdiag_mean': float(np.mean(off_diag)),
        'morf_offdiag_min': float(min(off_diag)),
        'morf_offdiag_max': float(max(off_diag)),
        'tp_offdiag_mean': float(np.mean(tp_off_diag)),
    },
}

# Save both locally and to Drive (Drive copy for persistence)
with open('/content/morfessor_transfer.json', 'w') as f:
    json.dump(out, f, indent=2)
try:
    drive_out = BASE_PATH + 'morfessor_transfer.json'
    with open(drive_out, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"\nSaved to: /content/morfessor_transfer.json")
    print(f"Saved to: {drive_out}")
except Exception as e:
    print(f"\nSaved to /content/morfessor_transfer.json (Drive save failed: {e})")